In [1]:
import numpy as np
from matplotlib import pyplot as plt
from numba import njit, prange
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import matplotlib as mpl
import pandas as pd
from sklearn.multiclass import OneVsRestClassifier as classifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.preprocessing import MultiLabelBinarizer, LabelBinarizer
from sklearn.metrics import log_loss, roc_curve
from sklearn.cluster import AgglomerativeClustering as clustering
import pandas as pd

In [2]:
#generate parameters for this model-- copy of august_curtis_params

max_sel = 0.1
gran=100
repeats = 20 #bear in mind each of these will be repeated, in simulations, 5 times-- so if we want 100 repeats we only need 20 here

base_sel = np.linspace(0, 0.1, num=gran+1)

selections = []

for sel in base_sel:
    for r in range(repeats):
        selections.append(sel)
    sel += gran

#remember each is repeated for 5 simulations within that-- because of the batching

In [3]:
#Extract mutational frequencies per sample.

filenames = ["11Aug_highest_death", "14Aug_highest_death", "14Aug_0.4"]
model_names= ["Model 1", "Model 2", "Model 3"]


batches = len(selections)
batch_size = 5

df = []

for (filename, model_name) in zip(filenames, model_names):
    stat_matrix = []
    for batch in range(batches):
        selection_level = selections[batch] #selection per batch
        for number in range(batch_size*batch+1, batch_size*(batch+1)+1): 
            sim_id = model_name + ":" + str(number)
            try:
                #Load in the output, which is a list of dictionaries.
                mut_dict = np.load(filename+"/exp_"+str(number)+"_mutdict.npy", allow_pickle=True)

                #Keep overall results.
                sim_df = []

                for sample_index, subdict in enumerate(mut_dict):
                    subdf = pd.DataFrame(list(subdict.items()), columns=['Mutation ID', 'CCF']).assign(
                        SimulationID = sim_id,
                        Model = model_name,
                        Sample = sample_index+1,
                        SelectionStrength = selection_level
                    )
                
                    sim_df += [subdf]
            
                #Attach all samples or none.
                df += sim_df

            except:
                print(filename, number, "failed")

#Put everything together and save.
df = pd.concat(df)
df.to_csv("sim-outputs.csv")



Load in and get a subset of the simulation outputs.

In [4]:
df = pd.read_csv("sim-outputs.csv", index_col=0)

df = df[(df['SelectionStrength']==0.0) | (df['SelectionStrength']==0.02) | (df['SelectionStrength']==0.1)]

df.to_csv("sim-outputs-subset.csv")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist

matplotlib.rcParams['font.family'] = 'Liberation Sans'
matplotlib.rcParams['font.size'] = 10

df = pd.read_csv('/mnt/user-data/uploads/sim-outputs-subset.csv')

MODELS = ['Model 1', 'Model 2', 'Model 3']
SELS   = [0.0, 0.02, 0.1]
SEL_LABELS = {
    0.0:  'Neutral  (s = 0)',
    0.02: 'Weak selection  (s = 0.02)',
    0.1:  'Strong selection  (s = 0.1)',
}
MODEL_COLORS = {'Model 1': '#2d6fa3', 'Model 2': '#2a8a62', 'Model 3': '#c0562a'}
SEL_COLORS   = {0.0: '#6b6b64', 0.02: '#3266ad', 0.1: '#b83030'}

N_SIMS     = 5
CCF_THRESH = 0.05

cmap_colors = [
    (0.96, 0.95, 0.93),
    (0.84, 0.91, 0.96),
    (0.28, 0.60, 0.74),
    (0.06, 0.32, 0.48),
]
ccf_cmap = LinearSegmentedColormap.from_list('ccf',
    list(zip([0.0, 0.08, 0.5, 1.0], cmap_colors)))

def cluster_cols(mat):
    if mat.shape[1] <= 2:
        return np.arange(mat.shape[1])
    try:
        dist = pdist(mat.T, metric='euclidean')
        link = linkage(dist, method='average')
        return leaves_list(link)
    except Exception:
        return np.arange(mat.shape[1])

def sim_feature_vector(pivot):
    det = pivot > CCF_THRESH
    sharing = det.sum(axis=1)
    n_s = det.shape[1]
    per_sample_mean = pivot[det].mean(axis=0).fillna(0).values
    per_sample_frac = det.mean(axis=0).values
    frac_clonal  = (sharing == n_s).mean()
    frac_private = (sharing == 1).mean()
    frac_shared  = ((sharing > 1) & (sharing < n_s)).mean()
    return np.concatenate([per_sample_mean, per_sample_frac,
                           [frac_clonal, frac_private, frac_shared]])

print('Loading data...')
examples = {}
for (model, sel), grp in df.groupby(['Model', 'SelectionStrength']):
    sim_sizes = grp.groupby('SimulationID')['Mutation ID'].nunique().sort_values()
    raw = []
    for sim_id in sim_sizes.index:
        sub   = grp[grp['SimulationID']==sim_id]
        pivot = sub.pivot_table(
            index='Mutation ID', columns='Sample', values='CCF', fill_value=0)
        passes  = (pivot > CCF_THRESH).any(axis=1)
        pivot_f = pivot.loc[passes]
        n_total    = int(sim_sizes[sim_id])
        n_filtered = len(pivot_f)
        feat = sim_feature_vector(pivot_f) if n_filtered > 0 else np.zeros(19)
        if n_filtered > 0:
            pivot_f = pivot_f.loc[pivot_f.mean(axis=1).sort_values(ascending=False).index]
            col_order = cluster_cols(pivot_f.values)
            pivot_f   = pivot_f.iloc[:, col_order]
        raw.append({
            'n_total':    n_total,
            'n_filtered': n_filtered,
            'matrix':     pivot_f.values if n_filtered > 0 else np.zeros((1, 8)),
            'feat':       feat,
        })
    feats = np.array([r['feat'] for r in raw])
    try:
        order = leaves_list(linkage(pdist(feats, metric='euclidean'), method='ward'))
    except Exception:
        order = np.arange(len(raw))
    idxs = np.linspace(0, len(order)-1, N_SIMS, dtype=int)
    examples[(model, float(sel))] = [raw[order[i]] for i in idxs]
    print(f'  {model} s={sel}: done')

print('Building figure...')

FIG_W = 18
FIG_H = 10   # taller to give title space

# heatmap area — leave generous room at top for titles + col headers
OL   = 0.065   # left edge of heatmap area
OR   = 0.910   # right edge
OT   = 0.780   # top edge  ← pulled down to leave ~20% for titles/headers
OB   = 0.050   # bottom edge
OHGAP = 0.022  # gap between sel-strength blocks
OVGAP = 0.065  # gap between model blocks
IHGAP = 0.006  # gap between heatmaps within a block

n_outer_cols = 3
n_outer_rows = 3
outer_cell_w = (OR - OL - OHGAP * (n_outer_cols-1)) / n_outer_cols
outer_cell_h = (OT - OB - OVGAP * (n_outer_rows-1)) / n_outer_rows

inner_cols   = N_SIMS  # 3
inner_cell_w = (outer_cell_w - IHGAP * (inner_cols-1)) / inner_cols
inner_cell_h = outer_cell_h   # single row per block

fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor='white')

# place heatmap axes
all_axes = {}
for ri, model in enumerate(MODELS):
    for ci, sel in enumerate(SELS):
        ox0    = OL + ci * (outer_cell_w + OHGAP)
        oy_top = OT - ri * (outer_cell_h + OVGAP)
        for si in range(N_SIMS):
            ax_x0 = ox0 + si * (inner_cell_w + IHGAP)
            ax_y0 = oy_top - inner_cell_h
            ax    = fig.add_axes([ax_x0, ax_y0, inner_cell_w, inner_cell_h])
            all_axes[(model, sel, si)] = ax

# fill heatmaps
for ri, model in enumerate(MODELS):
    for ci, sel in enumerate(SELS):
        sims = examples[(model, sel)]
        for si, sim in enumerate(sims):
            ax  = all_axes[(model, sel, si)]
            mat = sim['matrix']
            n_t = sim['n_total']
            n_f = sim['n_filtered']
            pct = n_f / n_t * 100 if n_t > 0 else 0

            ax.imshow(mat, aspect='auto', cmap=ccf_cmap,
                      vmin=0, vmax=1, interpolation='nearest')

            ax.text(0.97, 0.02, f'{n_f:,} ({pct:.0f}%)',
                    transform=ax.transAxes, fontsize=6,
                    ha='right', va='bottom', color='white',
                    bbox=dict(boxstyle='round,pad=0.10',
                              facecolor=SEL_COLORS[sel],
                              edgecolor='none', alpha=0.82))

            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_linewidth(0.4)
                spine.set_color('#bbb')

# ── column headers (selection strength) — sit just above OT ─────
for ci, sel in enumerate(SELS):
    ox0 = OL + ci * (outer_cell_w + OHGAP)
    fig.text(ox0 + outer_cell_w/2, OT + 0.015,
             SEL_LABELS[sel],
             ha='center', va='bottom', fontsize=10, fontweight='bold',
             color=SEL_COLORS[sel])

# ── row labels (model) ───────────────────────────────────────────
for ri, model in enumerate(MODELS):
    oy_mid = OT - ri*(outer_cell_h+OVGAP) - outer_cell_h/2
    fig.text(OL - 0.012, oy_mid, model,
             ha='right', va='center', fontsize=10, fontweight='bold',
             color=MODEL_COLORS[model], rotation=90)

# ── main title + subtitle — well above the column headers ────────
fig.text(0.5, 0.97,
         'Tumour simulations — CCF across multi-region samples'
         '  (mutations with CCF\u202f<\u202f5% in all samples removed)',
         ha='center', va='top', fontsize=10, fontweight='bold', color='#1a1a1a')
fig.text(0.5, 0.93,
         '5 representative simulations per group  ·  '
         'rows\u202f=\u202fmutations sorted by mean CCF (high\u202f→\u202flow)  ·  '
         'columns\u202f=\u202fsamples clustered by CCF similarity  ·  '
         'badge\u202f=\u202fmutations retained (% of total)',
         ha='center', va='top', fontsize=8, color='#555')

# ── colorbar ─────────────────────────────────────────────────────
cbar_ax = fig.add_axes([OR+0.015, OB+0.03, 0.012, OT-OB-0.06])
sm = plt.cm.ScalarMappable(cmap=ccf_cmap, norm=mcolors.Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
cbar.set_ticklabels(['0', '0.25', '0.50', '0.75', '1.0'])
cbar.ax.tick_params(labelsize=8, length=2.5, width=0.6)
cbar.outline.set_linewidth(0.5)
cbar.ax.set_title('CCF', fontsize=9, pad=5, color='#333')

print('Saving...')
plt.savefig('/mnt/user-data/outputs/tumour_sims_5x_arial.pdf',
            bbox_inches='tight', facecolor='white')
print('Done.')